# Lección 11.4 — LSTM y GRU: puertas para recordar y olvidar

> **Módulo 11 · Secuencias y lenguaje** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 9.3 (sigmoide y tanh), Lección 9.7 (gradientes que desaparecen), Lección 10.4 (conexiones residuales), Lección 11.3

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía cómo las **puertas** de una LSTM deciden qué olvidar, qué guardar y qué mostrar.
- Explicar por qué la **celda de memoria** deja pasar el gradiente sin que se desvanezca.
- Comparar **RNN, LSTM y GRU** en una tarea de memoria a largo plazo y en parámetros.
- Decidir cuándo vale la pena una LSTM/GRU y cuándo basta una RNN simple.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams["figure.figsize"] = (9, 3.5)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: la cinta transportadora con tres porteros

En la Lección 11.3 la RNN **reescribía todo su post-it** en cada paso: lo viejo se diluía sí o sí.

Una **LSTM** (*Long Short-Term Memory*, "memoria de corto plazo… larga") cambia el post-it por una **cinta transportadora** que atraviesa toda la secuencia. Lo que pones en la cinta **viaja sin cambios** de paso en paso, salvo que alguien decida tocarlo. Y hay **tres porteros** (puertas, *gates*) que deciden:

```
                 🚪 OLVIDO            🚪 ENTRADA
                 "¿qué tiro?"         "¿qué agrego?"
 cinta  cₜ₋₁ ════════(×)═══════════════(+)══════════════► cₜ   (la memoria de largo plazo)
                                        ▲                │
                                  nueva información      ▼
                                                    🚪 SALIDA "¿qué muestro ahora?" ──► hₜ
```

| Puerta | Pregunta que responde | Ejemplo al leer un texto |
|---|---|---|
| 🚪 **Olvido** (*forget*) | ¿Qué parte de la memoria ya no sirve? | Empieza una frase con otro sujeto → olvida el anterior |
| 🚪 **Entrada** (*input*) | ¿Qué de lo nuevo vale la pena guardar? | Apareció el sujeto "los niños" (plural) → guárdalo |
| 🚪 **Salida** (*output*) | ¿Qué parte de la memoria necesito **ahora**? | Toca escribir el verbo → saca "era plural" |

Cada puerta es un número entre **0** (cerrada) y **1** (abierta), calculado con una sigmoide (Lección 9.3) a partir de la entrada y de la memoria anterior. **La red aprende cuándo abrir y cerrar cada puerta.**

### ✋ Ejercicio 1 — Sé el portero
Lees: *"Ana compró pan. Luego **Luis** fue al cine y **él**…"*. Al llegar a "Luis", ¿qué deberían hacer la puerta de **olvido** y la de **entrada** con la información "el personaje es Ana"? ¿Y al llegar a "él"?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- En **"Luis"**: la puerta de olvido se **cierra parcialmente** sobre "personaje actual = Ana" (ya no es quien actúa) y la de entrada se **abre** para guardar "personaje actual = Luis".
- En **"él"**: la puerta de **salida** se abre para mostrar "el personaje actual es Luis (masculino, singular)", que ayuda a interpretar el pronombre y a elegir el verbo.

Nadie programa esas reglas: la red aprende a manejar las puertas porque eso reduce el error.
</details>

---
## 2. 📖 Cómo funciona por dentro (a mano)

Las dos ecuaciones clave de la LSTM son:

```
cₜ = f · cₜ₋₁  +  i · c̃ₜ          ← la cinta: (olvido × lo de antes) + (entrada × lo nuevo)
hₜ = o · tanh(cₜ)                ← lo que se muestra: (salida × la memoria)
```

- **f**, **i**, **o** son las puertas (entre 0 y 1).
- **c̃ₜ** es la "información candidata" que propone el paso actual (como la RNN simple: una `tanh` de la entrada y de hₜ₋₁).

Fíjate en la primera ecuación: si **f = 1** e **i = 0**, entonces **cₜ = cₜ₋₁**. ¡La memoria pasa **intacta**! Comparemos con la RNN de la Lección 11.3.

In [ ]:
def celda_lstm_a_mano(c_inicial, pasos, f, i, c_nuevo=0.0):
    c = c_inicial
    for _ in range(pasos):
        c = f * c + i * c_nuevo           # la cinta: olvido × lo anterior + entrada × lo nuevo
    return c

def rnn_a_mano(h_inicial, pasos, w_h):
    h = h_inicial
    for _ in range(pasos):
        h = np.tanh(w_h * h)              # la RNN de la 11.3, recibiendo solo ceros
    return h

print(f"RNN  (w_h=0.9)        tras 25 pasos: {rnn_a_mano(0.8, 25, w_h=0.9):.4f}")
print(f"LSTM (f=1.0, i=0.0)   tras 25 pasos: {celda_lstm_a_mano(0.8, 25, f=1.0, i=0.0):.4f}")

### 🔮 Ejercicio 2 — Juega con las puertas
Predice qué valor tendrá la memoria de la LSTM tras 25 pasos, empezando en 0.8, si:
- a) `f = 0.9`, `i = 0` (olvida un poquito en cada paso)
- b) `f = 0`, `i = 1`, `c_nuevo = 0.3` (borra todo y escribe lo nuevo)

✍️ *Mis predicciones:*

In [ ]:
print(f"a) f=0.9, i=0          : {celda_lstm_a_mano(0.8, 25, f=0.9, i=0.0):.4f}")
print(f"b) f=0,   i=1, nuevo=0.3: {celda_lstm_a_mano(0.8, 25, f=0.0, i=1.0, c_nuevo=0.3):.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **a)** 0.8 × 0.9²⁵ ≈ **0.057**: un olvido pequeño, repetido 25 veces, se come casi toda la memoria. Para recordar mucho tiempo, la red tiene que aprender a dejar la puerta de olvido **muy** cerca de 1.
- **b)** **0.3**: la puerta de olvido en 0 **borra** lo anterior y la de entrada en 1 escribe lo nuevo. Esto es lo que la RNN simple hace **siempre**, quiera o no.

La LSTM puede elegir **en cada paso** entre conservar y reescribir. La RNN simple no tiene esa opción.
</details>

### 📖 ¿Por qué esto arregla el gradiente?
En la RNN, la memoria pasa por `tanh(w_h · h)` en cada paso: el gradiente se **multiplica** por pesos y derivadas de `tanh` una y otra vez, y se desvanece.

En la LSTM, la cinta se actualiza con una **suma**: `cₜ = f · cₜ₋₁ + …`. El gradiente que viaja hacia atrás por la cinta se multiplica solo por **f**. Si la red aprende f ≈ 1, el gradiente **viaja casi intacto** por cientos de pasos.

> 💡 **¿Te suena?** Es la misma idea que las **conexiones residuales** de ResNet (Lección 10.4): un "atajo" donde la información se **suma** en lugar de transformarse, para que el gradiente fluya. La LSTM (1997) la usó casi 20 años antes, a lo largo del tiempo.

### ✋ Ejercicio 3
Si la puerta de olvido valiera siempre **0.5**, ¿la LSTM tendría memoria larga? ¿Qué número multiplicaría al gradiente tras 20 pasos?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Tras 20 pasos el gradiente por la cinta se multiplicaría por 0.5²⁰ ≈ **0.000001**: igual de mal que la RNN. La LSTM no tiene memoria larga "gratis"; tiene la **posibilidad** de tenerla si aprende a dejar f ≈ 1. Por eso, al crear una LSTM, se suele inicializar la puerta de olvido **abierta** (Sección 5).
</details>

---
## 3. 📖 GRU: la versión compacta

La **GRU** (*Gated Recurrent Unit*, 2014) simplifica la LSTM:
- Junta la cinta y el post-it en **una sola memoria** hₜ.
- Usa **dos** puertas en vez de tres:
  - 🚪 **Actualización** (*update*, z): ¿cuánto de la memoria vieja **conservo** y cuánto **reemplazo** por lo nuevo? (hace el trabajo de las puertas de olvido y de entrada a la vez)
  - 🚪 **Reinicio** (*reset*, r): ¿cuánto de la memoria vieja uso para **proponer** lo nuevo?

```
hₜ = z · hₜ₋₁  +  (1 − z) · h̃ₜ      ← si z = 1: conserva todo; si z = 0: reemplaza todo
```

| | RNN simple | LSTM | GRU |
|---|---|---|---|
| Memorias | 1 (h) | 2 (h y la cinta c) | 1 (h) |
| Puertas | 0 | 3 (olvido, entrada, salida) | 2 (actualización, reinicio) |
| Memoria larga | ❌ | ✅ | ✅ |
| Parámetros (mismo tamaño) | 1× | ≈4× | ≈3× |

### 🔮 Ejercicio 4 — Cuenta los parámetros
Creamos las tres redes con la **misma** entrada (10 números) y la **misma** memoria (32 neuronas). ¿Cuántos parámetros esperas en cada una, en proporción a la RNN?

✍️ *Mi predicción:*

In [ ]:
for tipo in ["RNN", "LSTM", "GRU"]:
    red = getattr(nn, tipo)(input_size=10, hidden_size=32, batch_first=True)
    print(f"{tipo:>4}: {sum(p.numel() for p in red.parameters()):>6,} parámetros")

lstm = nn.LSTM(10, 32, batch_first=True)
salidas, (h_final, c_final) = lstm(torch.randn(4, 7, 10))
print("\nLa LSTM devuelve DOS memorias: h", tuple(h_final.shape), "y la cinta c", tuple(c_final.shape))

<details><summary><b>👀 Ver respuesta</b></summary>

RNN **1,408** · LSTM **5,632** (4×) · GRU **4,224** (3×). La LSTM hace 4 cálculos del tamaño de una RNN por paso (3 puertas + la información candidata) y la GRU, 3 (2 puertas + candidata). Más parámetros = más lenta y más datos para entrenar, pero con memoria larga.

Fíjate también en que `nn.LSTM` devuelve `(h, c)`: el post-it **y** la cinta.
</details>

---
## 4. 📖 La gran comparación: la prueba de memoria

Volvamos a la prueba de la Lección 11.3: la red ve un **número secreto** en el primer paso, luego **ruido**, y al final debe decir el secreto. Error 0 = recuerda perfecto; error ≈0.5 = no recuerda nada.

En la 11.3 la RNN simple falló con 25 pasos. Ahora pondremos a competir a las tres arquitecturas, en las **mismas condiciones**, y con **3 semillas** cada una (una sola semilla puede tener suerte, como un solo pliegue en la validación cruzada).

In [ ]:
def lote_memoria(n, largo, generador):
    x = torch.zeros(n, largo, 2)
    secreto = torch.rand(n, generator=generador) * 2 - 1
    x[:, 0, 0] = secreto                                              # canal 0: el secreto, solo en el paso 1
    x[:, 1:, 1] = torch.randn(n, largo - 1, generator=generador)     # canal 1: ruido que distrae
    return x, secreto

class Memorion(nn.Module):
    def __init__(self, tipo, oculto=32, sesgo_olvido=1.0):
        super().__init__()
        self.red = getattr(nn, tipo)(2, oculto, batch_first=True)
        self.salida = nn.Linear(oculto, 1)
        if tipo == "LSTM":   # puerta de olvido "abierta" al empezar (Sección 5)
            with torch.no_grad():
                self.red.bias_ih_l0[oculto:2 * oculto].fill_(sesgo_olvido)
                self.red.bias_hh_l0[oculto:2 * oculto].fill_(0.0)
    def forward(self, x):
        salidas, _ = self.red(x)
        return self.salida(salidas[:, -1]).squeeze(-1)

def prueba_de_memoria(tipo, largo, pasos=400, semilla=0):
    torch.manual_seed(semilla); gen = torch.Generator().manual_seed(semilla)
    modelo = Memorion(tipo)
    opt = torch.optim.Adam(modelo.parameters(), lr=0.005)
    for _ in range(pasos):
        x, secreto = lote_memoria(64, largo, gen)
        perdida = F.mse_loss(modelo(x), secreto)
        opt.zero_grad(); perdida.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        opt.step()
    x, secreto = lote_memoria(1000, largo, gen)
    with torch.no_grad():
        error = (modelo(x) - secreto).abs().mean().item()
    return error, modelo

### 🔮 Ejercicio 5 — ¿Quién recuerda?
Con el secreto **25 pasos** atrás, ¿cuántas de las 3 semillas crees que resolverá cada arquitectura (error < 0.1)?

✍️ *Mis predicciones:* RNN → _/3 · LSTM → _/3 · GRU → _/3

▶️ Esta celda tarda ≈15–60 s.

In [ ]:
LARGO = 25
modelos = {}
inicio = time.time()
for tipo in ["RNN", "LSTM", "GRU"]:
    errores = []
    for semilla in [0, 1, 2]:
        error, modelo = prueba_de_memoria(tipo, LARGO, semilla=semilla)
        errores.append(error)
        if semilla == 0: modelos[tipo] = modelo
    resueltas = sum(e < 0.1 for e in errores)
    print(f"{tipo:>4}: errores = {[round(e, 3) for e in errores]}  → resolvió {resueltas}/3")
print(f"({time.time() - inicio:.0f} s)")

<details><summary><b>👀 Ver respuesta</b></summary>

- **RNN:** errores ≈ 0.25, 0.01 y 0.49 → resuelve **1/3**, y con una sola semilla "con suerte".
- **LSTM:** errores ≈ 0.01–0.02 → **3/3**.
- **GRU:** errores ≈ 0.04, 0.006, 0.006 → **3/3**.

Con el mismo presupuesto de entrenamiento, las redes con puertas aprenden a **guardar el secreto en la cinta y cerrar la puerta al ruido**. La RNN a veces lo logra, pero es una lotería. (Los números exactos pueden variar un poco en tu máquina; el patrón no.)
</details>

### 🔮 Ejercicio 6 — Abramos el cerebro
En la Lección 11.3 medimos cuánto "siente" la salida final a cada entrada. Hagámoslo con los modelos **ya entrenados** (semilla 0): ¿cuánto cambia la respuesta si movemos un poquito **el secreto** del paso 1?

In [ ]:
gen = torch.Generator().manual_seed(123)
x, _ = lote_memoria(256, LARGO, gen)
for tipo, modelo in modelos.items():
    x_ = x.clone().requires_grad_(True)
    modelo(x_).sum().backward()
    sensibilidad = x_.grad[:, 0, 0].abs().mean().item()    # efecto del secreto (paso 1) en la respuesta
    print(f"{tipo:>4}: sensibilidad al paso 1 = {sensibilidad:.1e}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **RNN:** ≈ **1e-7**: el secreto prácticamente **no llega** al final. Lo que pasó en el paso 1 no influye.
- **LSTM** y **GRU:** ≈ **1** (es decir, 10 millones de veces más): si el secreto sube 0.1, la respuesta sube ≈0.1. ¡Lo llevan **intacto** por la cinta 25 pasos!

Este es el gradiente que desaparece (11.3) contra la cinta transportadora (Sección 2), medido en redes reales.
</details>

### ✏️ Ejercicio 7 — ¿Memoria infinita?
🔮 Antes de ejecutar: si alargamos la prueba a **50 pasos** con el mismo entrenamiento (400 pasos), ¿la LSTM y la GRU seguirán resolviéndola?

▶️ Tarda ≈20–30 s.

In [ ]:
LARGO_LARGO = 50      # ✏️ prueba 35, 50…
for tipo in ["LSTM", "GRU"]:
    error, _ = prueba_de_memoria(tipo, LARGO_LARGO, semilla=0)
    print(f"{tipo:>4} con {LARGO_LARGO} pasos: error = {error:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No:** con 50 pasos ambas se quedan en error ≈ **0.5**, como si no recordaran nada. Las puertas **amplían** mucho la memoria, pero no la hacen infinita: cuanto más lejos está la información, más difícil es **descubrir** (al entrenar) que hay que guardarla. Con más pasos de entrenamiento, más datos o trucos de inicialización, se puede llegar más lejos… hasta cierto punto.

Y siempre queda el **otro** problema de la 11.3: hay que pasar por los 50 pasos **uno tras otro**. Esa combinación (memoria limitada + lentitud) es lo que resolvió la **atención** (Módulo 12): en lugar de cargar todo en la cinta, el modelo **mira directamente** hacia atrás al paso que necesita.
</details>

---
## 5. 📖 Hiperparámetros y perillas

Las perillas de la RNN (Lección 11.3) siguen valiendo (`hidden_size`, `num_layers`, recorte de gradiente, largo de secuencia, `bidirectional`). Las LSTM/GRU añaden algunas:

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `hidden_size` | Tamaño de la memoria | Más capacidad; ×4 (LSTM) o ×3 (GRU) en parámetros | Memoria pobre | 64–512 |
| `num_layers` | LSTM apiladas | Más abstracción; más lento | Más simple | 1–3 |
| `dropout` (entre capas) | Apaga neuronas al azar entre capas (Lección 9.8) | Menos sobreajuste; aprende más lento | Más sobreajuste | 0.1–0.3 (solo si `num_layers` ≥ 2) |
| Sesgo inicial de la puerta de olvido | Qué tan "abierta" empieza la cinta | Recuerda más desde el principio | Olvida rápido al empezar | 1.0 (Keras lo pone por defecto; PyTorch, 0) |
| LSTM vs. GRU | Arquitectura | LSTM: algo más expresiva | GRU: más rápida, menos parámetros | Prueba GRU primero si tienes pocos datos |

### 🔮 Ejercicio 8 — La perilla escondida
Medimos cuánto "siente" una LSTM **recién creada** (sin entrenar) la entrada de hace 25 pasos, con tres valores del sesgo de la puerta de olvido. ¿Qué esperas al subirlo?

In [ ]:
for sesgo in [0.0, 1.0, 3.0]:        # ✏️
    torch.manual_seed(0)
    modelo = Memorion("LSTM", sesgo_olvido=sesgo)
    x = torch.randn(64, 25, 2, requires_grad=True)
    modelo(x).abs().sum().backward()
    print(f"sesgo de olvido = {sesgo}: sensibilidad al paso 1 = {x.grad[:, 0].abs().mean():.1e}")

<details><summary><b>👀 Ver respuesta</b></summary>

Sube muchísimo: ≈**2e-8** con sesgo 0, ≈**2e-5** con sesgo 1 y ≈**8e-3** con sesgo 3. Un sesgo positivo hace que la sigmoide de la puerta de olvido empiece cerca de 1 (sigmoide(3) ≈ 0.95), así que la cinta **arranca conservando** y el gradiente llega desde lejos desde el primer paso de entrenamiento. Por eso usamos 1.0 en la prueba de memoria. Es un truco pequeño con un efecto enorme.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| `ValueError: too many values to unpack` con LSTM | `nn.LSTM` devuelve `(salidas, (h, c))`, no `(salidas, h)` | Desempaqueta las dos memorias |
| La LSTM tarda en empezar a aprender dependencias largas | Puerta de olvido que arranca medio cerrada | Sesgo inicial de olvido ≈1 |
| LSTM grande que sobreajusta con pocos datos | ×4 parámetros respecto a una RNN | GRU, `hidden_size` menor, `dropout`, *early stopping* (Lección 7.2) |
| Entrenamiento lento con secuencias muy largas | Cálculo secuencial | Recortar secuencias, GRU, o Transformers (M12) |
| La LSTM no supera a la RNN | La tarea **no necesita** memoria larga | ¡Quédate con la más simple! (Práctica 9.3) |
| Pérdida `nan` | También pueden explotar | Recorte de gradiente, tasa de aprendizaje menor |

### 🔮 Ejercicio 9
Un compañero cambia su RNN por una LSTM para **predecir la siguiente letra de nombres de 5–10 letras**. ¿Esperas una gran mejora? ¿Por qué?

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Probablemente no una gran mejora.** Para predecir la siguiente letra de un nombre corto basta con recordar las últimas letras: la memoria corta de la RNN casi alcanza. La LSTM tendrá unas 3–4 veces más parámetros (más lenta) y una mejora pequeña. Lo comprobarás tú mismo en la práctica integradora.
</details>

---
## 7. 📋 Fichas de los modelos

| | LSTM | GRU |
|---|---|---|
| **Tipo** | Red recurrente con puertas | Red recurrente con puertas |
| **Intuición** | Cinta transportadora + 3 porteros (olvido, entrada, salida) | Una memoria + 2 porteros (actualización, reinicio) |
| **Cómo aprende** | BPTT; la cinta se actualiza sumando → el gradiente fluye | Igual, con la memoria mezclada por la puerta de actualización |
| **Hiperparámetros clave** | `hidden_size`, `num_layers`, `dropout`, sesgo de olvido | `hidden_size`, `num_layers`, `dropout` |
| **✅ Fortalezas** | Memoria larga (cientos de pasos) · Muy probada · Estable | Memoria larga · ≈25 % menos parámetros · Más rápida |
| **❌ Debilidades** | ×4 parámetros · Secuencial (lenta) · Memoria larga pero finita | Un poco menos expresiva en algunas tareas · Secuencial |
| **Úsalo cuando…** | Series de tiempo, sensores, texto mediano, voz en dispositivos | Igual que la LSTM, sobre todo con pocos datos o poco cómputo |
| **Evítalo cuando…** | Hay muchísimo texto y GPU: los Transformers ganan | Igual |

## 8. 🧠 Resumen en 5 puntos
1. Las **LSTM** añaden una **cinta** (celda de memoria) y **tres puertas** que deciden qué olvidar, qué guardar y qué mostrar.
2. La cinta se actualiza con una **suma**: si la puerta de olvido ≈ 1, la memoria **y el gradiente** viajan intactos (la misma idea que las conexiones residuales).
3. La **GRU** hace lo mismo con **dos** puertas y una sola memoria: ≈3× los parámetros de una RNN (la LSTM, ≈4×).
4. En una prueba de memoria de 25 pasos, LSTM y GRU la resuelven siempre; la RNN simple, casi nunca. Pero su memoria **no es infinita** (a 50 pasos fallan con el mismo entrenamiento).
5. Si la tarea solo necesita memoria corta, una RNN simple rinde igual y es más barata. Para dependencias muy largas y mucho texto, la respuesta moderna es la **atención** (M12).

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué puerta de la LSTM decide qué parte de la memoria se **descarta**?
- a) Entrada · b) Olvido · c) Salida · d) Reinicio

**2.** Si la puerta de olvido vale 1 y la de entrada vale 0 durante 100 pasos, la cinta…
- a) Se borra · b) Se conserva intacta · c) Explota · d) Se vuelve 0.5

**3.** Con la misma entrada y `hidden_size`, ¿cuál tiene **menos** parámetros?
- a) LSTM · b) GRU · c) RNN simple · d) Todas igual

**4.** ¿Qué idea comparten la cinta de la LSTM y las conexiones residuales de ResNet?
- a) Ambas usan convoluciones · b) La información se **suma** por un atajo, así que el gradiente fluye sin desvanecerse · c) Ambas son bidireccionales · d) Ninguna

**5.** Tu LSTM y tu RNN dan el mismo error de validación prediciendo el siguiente paso de una señal de ritmo corto. ¿Qué eliges?
- a) La LSTM, porque es más moderna · b) La RNN: mismo resultado, menos parámetros y más rápida

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** (La de reinicio es de la GRU.)
2. **b)** cₜ = 1 · cₜ₋₁ + 0 · lo nuevo.
3. **c)** RNN 1× · GRU ≈3× · LSTM ≈4×.
4. **b)** Atajos aditivos para el gradiente.
5. **b)** A igual desempeño, el modelo más simple (Lección 1.6).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Explícale a alguien sin conocimientos técnicos, en 3 frases y sin fórmulas, por qué una LSTM recuerda más que una RNN simple.

**B. Caso de análisis 💼** Una empresa de mantenimiento predictivo tiene sensores de vibración que miden 100 veces por segundo. Las fallas graves suelen anunciarse con una pequeña anomalía **unos 2 minutos antes**. ¿Cuántos pasos de memoria necesitaría una LSTM que lea la señal cruda? ¿Qué propondrías para hacerle la vida más fácil?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** *"Una red simple lee como alguien que reescribe toda su libreta con cada palabra: al final, lo del principio se perdió. Una LSTM tiene una libreta aparte donde lo importante se queda escrito y unos 'porteros' que deciden cuándo borrar y cuándo anotar algo nuevo. Así puede acordarse de algo que leyó hace mucho, si aprendió que era importante."*

**B.** 2 minutos × 60 s × 100 lecturas/s = **12,000 pasos**: demasiado, incluso para una LSTM. Propuestas: (1) **resumir** la señal en ventanas (p. ej., cada segundo: media, máximo, energía) → 120 pasos, al alcance de una LSTM/GRU; (2) una red **convolucional 1D** (Módulo 10) que comprime la señal antes de la LSTM; (3) atención/Transformers (M12) sobre los resúmenes. Siempre con validación temporal (11.1).
</details>

### 9.3 🧩 Práctica integradora: ¿vale la pena la LSTM para inventar nombres?

En la Lección 11.3 entrenaste una RNN de caracteres que inventa nombres. Ahora harás de **científico/a escéptico/a**: ¿mejoraría con LSTM o GRU? Esta vez medirás con datos **que el modelo no vio**.

**Paso 1 ▶️ — Separa nombres para validar.** Reservamos 36 nombres (20 %) que la red nunca verá al entrenar.

In [ ]:
from torch.nn.utils.rnn import pad_sequence
import random

NOMBRES = """
ana andrea adriana alejandra alicia alma amalia amparo angela antonia araceli ariana
aurora beatriz blanca brenda camila carla carmen carolina catalina cecilia claudia clara
cristina daniela debora diana dolores elena elisa elvira emilia esperanza estela eva
fabiola fernanda flor gabriela gloria graciela guadalupe ines irene isabel jimena josefina
juana julia julieta karla laura leticia liliana lorena lucia luisa luz magdalena manuela
marcela margarita maria mariana marina marta mercedes micaela miriam monica natalia noemi
norma olga paola patricia paula pilar raquel rebeca regina rocio rosa rosario sandra sara
silvia sofia soledad susana teresa valentina valeria vanesa veronica victoria ximena
yolanda adrian agustin alberto alejandro alfonso alfredo alvaro andres angel antonio
armando arturo benjamin bernardo carlos cesar cristian daniel david diego eduardo emilio
enrique ernesto esteban fabian federico felipe fernando francisco gabriel gerardo gonzalo
guillermo gustavo hector hugo ignacio ivan jaime javier jesus joaquin jorge jose juan
julian julio leonardo lorenzo lucas luis manuel marco mario martin mateo mauricio miguel
nicolas oscar pablo pedro rafael ramiro raul ricardo roberto rodrigo rolando ruben
salvador samuel santiago sebastian sergio simon tomas ulises vicente victor
""".split()
alfabeto = ["."] + sorted(set("".join(NOMBRES)))
c2i = {c: i for i, c in enumerate(alfabeto)}
V = len(alfabeto)

def preparar(lista):
    datos = pad_sequence([torch.tensor([c2i[c] for c in "." + n + "."]) for n in lista],
                         batch_first=True, padding_value=-1)
    return datos[:, :-1].clamp(min=0), datos[:, 1:]

random.seed(0)
mezclados = random.sample(NOMBRES, len(NOMBRES))
validacion, entrenamiento = mezclados[:36], mezclados[36:]
X_tr, Y_tr = preparar(entrenamiento)
X_va, Y_va = preparar(validacion)
print(f"Entrenamiento: {len(entrenamiento)} nombres · Validación: {len(validacion)} (p. ej. {validacion[:5]})")

✋ Aquí sí barajamos antes de separar. ¿No era eso lo prohibido en la Lección 11.1?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Lo prohibido es barajar **los pasos de una misma serie de tiempo**. Aquí cada **nombre** es una secuencia independiente de las demás (como cada foto de gato): barajar nombres está bien. Lo que **nunca** barajamos es el orden de las letras **dentro** de cada nombre.
</details>

**Paso 2 ▶️ — Entrena las tres arquitecturas** con lo mismo: 300 épocas, misma semilla, mismo tamaño de memoria. Registramos la pérdida de validación en cada época. (≈5–15 s)

In [ ]:
class Inventor(nn.Module):
    def __init__(self, tipo, oculto=64):
        super().__init__()
        self.emb = nn.Embedding(V, 16)
        self.red = getattr(nn, tipo)(16, oculto, batch_first=True)
        self.salida = nn.Linear(oculto, V)
    def forward(self, x):
        salidas, _ = self.red(self.emb(x))
        return self.salida(salidas)

def perdida_de(modelo, X, Y):
    return F.cross_entropy(modelo(X).reshape(-1, V), Y.reshape(-1), ignore_index=-1)

historias = {}
for tipo in ["RNN", "LSTM", "GRU"]:
    torch.manual_seed(0)
    modelo = Inventor(tipo)
    opt = torch.optim.Adam(modelo.parameters(), lr=0.005)
    inicio, hist = time.time(), []
    for epoca in range(300):
        perdida = perdida_de(modelo, X_tr, Y_tr)
        opt.zero_grad(); perdida.backward(); opt.step()
        with torch.no_grad():
            hist.append((perdida.item(), perdida_de(modelo, X_va, Y_va).item()))
    historias[tipo] = np.array(hist)
    mejor = historias[tipo][:, 1].argmin()
    print(f"{tipo:>4}: {sum(p.numel() for p in modelo.parameters()):>6,} parámetros · {time.time() - inicio:4.1f} s · "
          f"mejor validación = {historias[tipo][mejor, 1]:.2f} en la época {mejor + 1}")

🔮 Antes de graficar: ¿cuál crees que ganó? ¿Por mucho?

✍️ *Mi apuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Casi empate:** la RNN llega a una pérdida de validación mínima de ≈**2.20**; la LSTM y la GRU, a ≈**2.13**. Es una mejora pequeña (≈3 %) a cambio de ≈3× (LSTM) y ≈2.4× (GRU) los parámetros y de entrenar más lento. Para nombres cortos, la memoria corta casi alcanza (Ejercicio 9).
</details>

**Paso 3 ▶️ — Mira las curvas** (Lección 3.7).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for tipo, h in historias.items():
    ax[0].plot(h[:, 0], label=tipo); ax[1].plot(h[:, 1], label=tipo)
ax[0].set_title("Pérdida de ENTRENAMIENTO"); ax[1].set_title("Pérdida de VALIDACIÓN (nombres no vistos)")
for a in ax: a.set_xlabel("época"); a.legend()
plt.show()

✋ Describe lo que ves en la gráfica de validación. ¿Qué diagnóstico le das? ¿Cuántas épocas habrías entrenado?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

La pérdida de entrenamiento baja sin parar (hasta ≈0.7), pero la de validación toca fondo (la RNN hacia la **época 30**, la LSTM y la GRU hacia la **65–80**) y luego **sube** (hasta ≈3.3–3.6): **sobreajuste** de manual, en las tres arquitecturas. Las redes memorizan los 144 nombres de entrenamiento. Habría que parar en el mínimo de cada una (*early stopping*, Lección 7.2). Fíjate en que la RNN, más pequeña y rápida, también **empieza a memorizar antes**. ¡Las 150 épocas que usamos en la Lección 11.3 ya eran demasiadas para generalizar! (Allí lo intuiste: con 300 épocas casi solo repetía nombres de la lista.)
</details>

**Paso 4 ✏️ — Tu veredicto.** Completa la tabla y escribe tu recomendación en dos frases.

| | RNN | LSTM | GRU |
|---|---|---|---|
| Parámetros | | | |
| Mejor pérdida de validación | | | |
| Época del mínimo | | | |

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver un ejemplo</b></summary>

| | RNN | LSTM | GRU |
|---|---|---|---|
| Parámetros | ≈7,400 | ≈23,100 | ≈17,900 |
| Mejor pérdida de validación | ≈2.20 | ≈2.13 | ≈2.13 |
| Época del mínimo | ≈30 | ≈80 | ≈65 |

*"Para inventar nombres cortos, la GRU con early stopping (≈65 épocas) es un buen punto medio: iguala a la LSTM con menos parámetros y mejora un poco a la RNN simple. Si el costo importara mucho (un celular, por ejemplo), la RNN simple rinde casi igual con un tercio de los parámetros. Si quisiéramos generar textos largos, ahí sí la memoria de la GRU/LSTM, o un Transformer, marcaría la diferencia."*

(Cualquier recomendación es válida si se apoya en estos números; lo importante es **no** elegir la LSTM "porque es más moderna".)
</details>

---
## 📚 Para profundizar
- 📘 **Artículo:** Christopher Olah, *Understanding LSTM Networks* (2015): los diagramas de LSTM más famosos de internet (inglés; hay traducciones).
- 📘 **Artículo:** Jozefowicz, Zaremba y Sutskever (2015), *An Empirical Exploration of Recurrent Network Architectures*: de donde viene el truco del sesgo de olvido en 1.
- 📺 **Video:** StatQuest, "Long Short-Term Memory (LSTM), Clearly Explained" (inglés, con subtítulos).

---
⬅️ **Anterior:** [11.3 Redes recurrentes (RNN) y su memoria corta](11.3-redes-recurrentes.ipynb) · ➡️ **Siguiente:** [11.5 Seq2seq y el cuello de botella](11.5-seq2seq-y-cuello-de-botella.ipynb)